In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# RQ1 Part 2: Full UMLS Pool Semantic Entropy

**Depends on Part 1 artifacts only.** This notebook does **not** regenerate
perturbations, run Marian back-translation, call `.generate()`, or load
generative LLMs.

Required input:
- `outputs/rq1/intermediate/rq1_model_outputs.csv` (encoder outputs + metadata from Part 1)

Also used:
- `outputs/rq1/tables/rq1_entropy_scores.csv` (Part 1 mesh-pool entropy for side-by-side comparison)
- Setup-loaded `cui_pool` from `~/data/umls/pools/cui_pool_full_umls.pkl`

Run the Setup cell first, then P2.1 → P2.6 in order.


In [ ]:
# Setup (Part 2): keep byte-identical across RQ notebooks
import json
import os
import pickle
from pathlib import Path

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

CONFIG_PATH = (PROJECT_ROOT / "config" / "config.json")
with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)

# PROJECT_ROOT must be the REPO ROOT (contains scripts/ AND config/), never <repo>/config.
# The config/ restructure broke the old `CONFIG_PATH.parent` (it pointed at config/), which
# made `from scripts...` fail and mis-rooted INTER_DIR/outputs. Anchor robustly by walking up.
def _repo_root_anchor(start: Path) -> Path:
    for _p in [start.resolve(), *start.resolve().parents]:
        if (_p / "scripts").is_dir() and (_p / "config" / "config.json").is_file():
            return _p
    raise FileNotFoundError(f"repo root (with scripts/ and config/config.json) not found from {start}")

PROJECT_ROOT = _repo_root_anchor(CONFIG_PATH.parent)


def _expand_tree(obj):
    """Expand ~ in all string paths; leave non-strings / null unchanged."""
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _model_src(key: str) -> str:
    """Return local path or HF hub id for a model key in CFG['models']."""
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def load_generative(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    return tokenizer, model


def load_encoder(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModel.from_pretrained(src)
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = cui_pool.get("n_cuis", len(cui_pool.get("cuis", {})))
print(f"Config: {CONFIG_PATH}")
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
if _n_cuis < 10_000:
    print("WARNING: CUI count looks like the MeSH subset (~1,201), not full UMLS.")
else:
    print("Confirmed: full_umls-scale CUI pool loaded.")


# PART 2: Full UMLS Pool Semantic Entropy (Sensitivity / Depth Analysis)

**Do not modify Part 1 cells above.** This section re-assigns CUIs using the full
UMLS 2026AA pool (~3.34M CUIs) via a SapBERT + FAISS embedding index, then
recomputes semantic entropy for comparison with the Part 1 MeSH-linker pool
(~1,201 CUIs).

| Setting | Role |
|---|---|
| Primary | Full vocabulary + `MIN_FORM_LEN` filter (default 3) |
| Sensitivity | `PREFERRED_ONLY=True` keeps TTY ∈ {PT, PN} only |

Model **inference is not re-run**. Inputs come from `outputs/rq1/intermediate/rq1_model_outputs.csv`.
CUI assignment uses SapBERT retrieval + the five-rule disambiguation protocol
(exact match → contextual cosine → ST21pv filter → confidence threshold → frequency tiebreak).


## P2.1) Pool preparation (length filter + preferred-only sensitivity flag)

Loads `cui_pool` from the Setup cell (full_umls). Builds the active `(CUI, surface_form)`
list for indexing, and prints sizes for **full / full+lenfilter / preferred-only**.


In [ ]:
# PART 2: Full-UMLS pool preparation
# Flags (primary config: PREFERRED_ONLY=False, MIN_FORM_LEN=3)
MIN_FORM_LEN = 3
PREFERRED_ONLY = False          # True = sensitivity analysis only (TTY in {PT, PN})
PREFERRED_TTYS = {"PT", "PN"}
CONFIDENCE_THRESHOLD = 0.70     # five-rule protocol
FAISS_TOP_K = int(CFG_YAML["umls"]["faiss_top_k"])  # locked 1000
SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
EMBED_BATCH_SIZE = 512

from collections import Counter, defaultdict
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

assert "cui_pool" in globals() and isinstance(cui_pool, dict), (
    "cui_pool missing — run the Setup cell first (full_umls pickle)."
)
assert cui_pool.get("pool_type") == "full_umls", (
    f"Expected pool_type=full_umls, got {cui_pool.get('pool_type')!r}"
)
_raw_cuis = cui_pool["cuis"]
print(
    f"Loaded cui_pool: type={cui_pool.get('pool_type')} | "
    f"CUIs={cui_pool.get('n_cuis'):,} | forms={cui_pool.get('n_surface_forms'):,}"
)

def _has_alpha(s: str) -> bool:
    return any(ch.isalpha() for ch in s)

# Single pass over pickle: full stats + length-filtered form->CUIs
_full_n_forms = 0
_full_cuis = set()
_len_form_to_cuis = defaultdict(set)
_cuis_with_kept_form = set()
_n_short = _n_noalpha = 0
_forms_seen_full = set()
_forms_seen_len = set()

for cui, rec in tqdm(_raw_cuis.items(), desc="Pool filter pass", unit=" CUI"):
    _full_cuis.add(cui)
    for form in rec.get("surface_forms", ()):
        s = str(form).strip()
        _full_n_forms += 1
        _forms_seen_full.add(s)
        if len(s) < MIN_FORM_LEN:
            _n_short += 1
            continue
        if not _has_alpha(s):
            _n_noalpha += 1
            continue
        _len_form_to_cuis[s].add(cui)
        _forms_seen_len.add(s)
        _cuis_with_kept_form.add(cui)

_unreachable = _full_cuis - _cuis_with_kept_form
print(f"[full]              forms={len(_forms_seen_full):,} | CUIs={len(_full_cuis):,} "
      f"(raw form refs={_full_n_forms:,})")
print(f"[full+lenfilter]    forms={len(_forms_seen_len):,} | CUIs={len(_cuis_with_kept_form):,}")
print(f"  dropped short(<{MIN_FORM_LEN}): {_n_short:,} | no-alpha: {_n_noalpha:,}")
print(f"  forms removed: {len(_forms_seen_full) - len(_forms_seen_len):,}")
print(f"  CUIs losing ALL surface forms (unreachable): {len(_unreachable):,}")
del _forms_seen_full
gc.collect()

# Preferred-only (TTY in {PT, PN}) via MRCONSO: form-level TTY
_umls_meta = Path(
    CFG.get("umls_meta", str(Path.home() / "data/umls/2026AA/2026AA/META"))
).expanduser()
_mrconso = _umls_meta / "MRCONSO.RRF"
_pref_form_to_cuis = defaultdict(set)
_pref_n_raw = 0
if _mrconso.is_file():
    print(f"Streaming preferred-only forms from {_mrconso} ...")
    with open(_mrconso, "r", encoding="utf-8", errors="replace") as _fh:
        for _line in tqdm(_fh, desc="MRCONSO preferred", unit=" lines"):
            p = _line.rstrip("\n").split("|")
            if len(p) <= 16:
                continue
            if p[1] != "ENG" or p[16] != "N":
                continue
            if p[12] not in PREFERRED_TTYS:
                continue
            s = p[14].strip()
            _pref_n_raw += 1
            if len(s) < MIN_FORM_LEN or not _has_alpha(s):
                continue
            _pref_form_to_cuis[s].add(p[0])
    print(
        f"[preferred-only]    forms={len(_pref_form_to_cuis):,} | "
        f"CUIs={len({c for cuis in _pref_form_to_cuis.values() for c in cuis}):,} "
        f"(raw PT/PN rows={_pref_n_raw:,}, after lenfilter)"
    )
else:
    print(
        f"[preferred-only]    MRCONSO not found at {_mrconso} — size unavailable "
        f"(will error if PREFERRED_ONLY=True)."
    )

# Active configuration
if PREFERRED_ONLY:
    if not _pref_form_to_cuis:
        raise RuntimeError("PREFERRED_ONLY=True but no preferred pairs available.")
    _form_to_cuis = _pref_form_to_cuis
    _pool_config_name = f"sapbert_pref_tty_len{MIN_FORM_LEN}"
else:
    _form_to_cuis = _len_form_to_cuis
    _pool_config_name = f"sapbert_full_len{MIN_FORM_LEN}"

_unique_forms = sorted(_form_to_cuis.keys())
_form_cui_pairs = [(c, f) for f, cuis in _form_to_cuis.items() for c in sorted(cuis)]

_cui_st21pv = {c: bool(rec.get("st21pv", False)) for c, rec in _raw_cuis.items()}
_cui_n_forms = {c: len(rec.get("surface_forms", ())) for c, rec in _raw_cuis.items()}

_exact_index = defaultdict(set)
for _f, _cuis in _form_to_cuis.items():
    _exact_index[_f.casefold()].update(_cuis)

print(f"\nACTIVE config: {_pool_config_name} | PREFERRED_ONLY={PREFERRED_ONLY}")
print(f"  unique surface forms to embed: {len(_unique_forms):,}")
print(f"  (CUI, form) pairs: {len(_form_cui_pairs):,}")
print("Size comparison:")
print(f"  full            forms={cui_pool.get('n_surface_forms'):,} CUIs={len(_full_cuis):,}")
print(f"  full+lenfilter  forms={len(_forms_seen_len):,} CUIs={len(_cuis_with_kept_form):,}")
print(
    f"  preferred-only  forms={len(_pref_form_to_cuis):,} | "
    f"CUIs={len({c for cuis in _pref_form_to_cuis.values() for c in cuis}):,}"
)


## P2.2) SapBERT embedding of surface forms (cached)

Embeds retained surface forms with `cambridgeltl/SapBERT-from-PubMedBERT-fulltext`
(GPU, fp16, L2-normalised). Cache key = pool configuration name under
`~/data/umls/embeddings/<config>/`.


In [ ]:
# PART 2: SapBERT embedding (cached)
_emb_root = Path.home() / "data" / "umls" / "embeddings" / _pool_config_name
_emb_root.mkdir(parents=True, exist_ok=True)
_emb_npy = _emb_root / "embeddings.npy"
_forms_json = _emb_root / "surface_forms.json"
_pairs_json = _emb_root / "cui_form_pairs.json"

_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {_device} | batch_size={EMBED_BATCH_SIZE}")

for _p in (_emb_npy, _forms_json, _pairs_json):
    assert _p.is_file(), (
        f"FIXED external input missing: {_p}. "
        "Do not rebuild embeddings.npy in a clean run."
    )
print(f"CACHE HIT — loading embeddings from {_emb_root} (rebuild disabled)")
_form_embeddings = np.load(_emb_npy, mmap_mode="r")  # mmap-only; do not copy/rebuild
with open(_forms_json, "r", encoding="utf-8") as _f:
    _cached_forms = json.load(_f)
with open(_pairs_json, "r", encoding="utf-8") as _f:
    _form_cui_pairs = [tuple(x) for x in json.load(_f)]
_unique_forms = _cached_forms
_recompute = False

if False:  # rebuild disabled — embeddings.npy is a fixed external input
    from transformers import AutoModel, AutoTokenizer

    print(f"Loading SapBERT: {SAPBERT_ID}")
    _tok = AutoTokenizer.from_pretrained(SAPBERT_ID)
    _mdl = AutoModel.from_pretrained(SAPBERT_ID)
    _mdl.to(_device)
    _mdl.eval()
    if _device == "cuda":
        _mdl.half()

    def _mean_pool(last_hidden, attn_mask):
        mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    _vectors = []
    with torch.no_grad():
        for _i in tqdm(range(0, len(_unique_forms), EMBED_BATCH_SIZE),
                       desc="SapBERT embed", unit="batch"):
            _batch = _unique_forms[_i:_i + EMBED_BATCH_SIZE]
            _enc = _tok(
                _batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
            )
            _enc = {k: v.to(_device) for k, v in _enc.items()}
            with torch.cuda.amp.autocast(enabled=(_device == "cuda")):
                _out = _mdl(**_enc)
                _pooled = _mean_pool(_out.last_hidden_state, _enc["attention_mask"])
                _pooled = torch.nn.functional.normalize(_pooled.float(), p=2, dim=1)
            _vectors.append(_pooled.detach().cpu().numpy().astype(np.float32))

    _form_embeddings = np.vstack(_vectors)
    assert _form_embeddings.shape[0] == len(_unique_forms)
    raise RuntimeError("refusing to write embeddings.npy")
    with open(_forms_json, "w", encoding="utf-8") as _f:
        json.dump(_unique_forms, _f)
    with open(_pairs_json, "w", encoding="utf-8") as _f:
        json.dump(_form_cui_pairs, _f)
    print(f"Saved embeddings: {_emb_npy} shape={_form_embeddings.shape}")

    del _mdl, _tok
    gc.collect()
    if _device == "cuda":
        torch.cuda.empty_cache()

# Map each unique form index -> list of CUIs
_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_c)
_form_index = {f: i for i, f in enumerate(_unique_forms)}

print(f"Ready: {_form_embeddings.shape[0]:,} L2-normalised form vectors "
      f"(dim={_form_embeddings.shape[1]})")


## P2.3) FAISS index (inner product = cosine on L2-normalised vectors)

Uses `IndexFlatIP` when memory allows; otherwise IVF with `nlist ≈ √N`.
Index is cached per pool configuration.


In [ ]:
# PART 2: FAISS index
try:
    import faiss
except ImportError as _e:
    raise ImportError(
        "faiss is required for full-UMLS retrieval. Install with:\n"
        '  pip install faiss-cpu\n'
        "or on GPU nodes: pip install faiss-gpu"
    ) from _e

_index_path = _emb_root / "faiss.index"
_n_vecs, _dim = _form_embeddings.shape
_bytes_needed = _n_vecs * _dim * 4  # float32
_mem_budget = 24 * (1024 ** 3)      # prefer FlatIP if under ~24GB raw vectors

assert _index_path.is_file(), (
    f"FIXED external input missing: {_index_path}. Do not rebuild FAISS in a clean run."
)
print(f"CACHE HIT — loading FAISS index from {_index_path} (rebuild disabled)")
_faiss_index = faiss.read_index(str(_index_path))
if False:  # rebuild disabled
    print(f"Building FAISS index over {_n_vecs:,} x {_dim} vectors "
          f"(~{_bytes_needed / 1e9:.1f} GB float32) ...")
    _xb = np.ascontiguousarray(_form_embeddings.astype(np.float32))
    if _bytes_needed <= _mem_budget:
        print("Using IndexFlatIP (exact cosine via inner product)")
        _faiss_index = faiss.IndexFlatIP(_dim)
        _faiss_index.add(_xb)
    else:
        _nlist = int(min(max(int(np.sqrt(_n_vecs)), 1024), 16384))
        print(f"Using IndexIVFFlat (nlist={_nlist}) — train then add")
        _quant = faiss.IndexFlatIP(_dim)
        _faiss_index = faiss.IndexIVFFlat(_quant, _dim, _nlist, faiss.METRIC_INNER_PRODUCT)
        # train on a subsample for speed if huge
        _train_n = min(_n_vecs, max(256_000, 40 * _nlist))
        _rng = np.random.default_rng(42)
        _train_idx = _rng.choice(_n_vecs, size=_train_n, replace=False)
        _faiss_index.train(_xb[_train_idx])
        _faiss_index.add(_xb)
        _faiss_index.nprobe = min(64, _nlist)
    raise RuntimeError("refusing to write faiss.index")
    print(f"Saved FAISS index -> {_index_path}")

print(f"FAISS ntotal={_faiss_index.ntotal:,} | type={type(_faiss_index).__name__}")
if hasattr(_faiss_index, "nprobe"):
    print(f"  nprobe={_faiss_index.nprobe}")


## P2.4) Mixed CUI / free-text assignment + entropy (8 models)

Reads `outputs/rq1/intermediate/rq1_all_model_outputs.csv`.

**Mapping** (only if `rq1_all_outputs_mapped.csv` is missing): detect **per row** whether
`output_text` is already a CUI (`^C\d+$` / `UMLS:C\d+`). Direct-CUI → keep;
free text → SapBERT+FAISS five-rule. `is_direct_cui` is **only** for that decision.

**Entropy / accuracy** (always, from the mapped file): treat every row identically —
use `predicted_cui` for all 8 models. Do **not** branch on `is_direct_cui`.

- \(\hat H = H / \log_2(m+1)\), \(m \ge 3\)
- accuracy: original-input `predicted_cui` vs `gold_cui_norm`
- `mapping_confidence`: mean original-input `confidence`

Writes `outputs/rq1/entropy_full_umls.csv`. Re-running entropy from a present mapped
cache needs **no GPU / SapBERT**.

In [ ]:
# PART 2: Mixed map (cached) + uniform entropy for all 8 models
# Mapping may use is_direct_cui to choose direct vs SapBERT.
# Entropy/accuracy ALWAYS use predicted_cui uniformly: never branch on is_direct_cui.

import gc
import re
import sys
from collections import Counter, defaultdict

from tqdm.auto import tqdm

UNASSIGNED = "UNASSIGNED"
_CUI_PAT = re.compile(r"^(?:UMLS:)?C\d+$", re.IGNORECASE)

INTER_DIR = PROJECT_ROOT / "outputs" / "rq1" / "intermediate"
_MODEL_OUT = INTER_DIR / "rq1_all_model_outputs.csv"

import sys as _sys
_sys.path.insert(0, str(PROJECT_ROOT))
from scripts.mm_assemble import assemble_partial_grid
from scripts.mm_shard_lib import grid_status, shard_root
_sh = shard_root(PROJECT_ROOT)
if (_sh / "shard_manifest.json").is_file():
    print("Assembling partial 8-model grid from complete instance-blocks...")
    print(grid_status(_sh))
    assemble_partial_grid(PROJECT_ROOT, require_all_eight=True)

assert _MODEL_OUT.is_file() and _MODEL_OUT.stat().st_size > 0, (
    f"Missing {_MODEL_OUT} — need at least one instance-block complete for all 8 models"
)
_PART1_ENT = PROJECT_ROOT / "outputs" / "rq1" / "tables" / "rq1_entropy_scores.csv"
_INST = INTER_DIR / "rq1_sampled_instances.csv"
assert _INST.is_file(), f"Missing {_INST} (do not copy the 550-row archive)"

OUT_MAPPED = INTER_DIR / "rq1_all_outputs_mapped.csv"
_out_dir = PROJECT_ROOT / "outputs" / "rq1"
_out_dir.mkdir(parents=True, exist_ok=True)
_out_csv = _out_dir / (
    "entropy_full_umls_preferred_only.csv" if PREFERRED_ONLY else "entropy_full_umls.csv"
)

# pool_config name may come from P2.1; fall back if running entropy-only from cache
if "_pool_config_name" not in globals():
    _pool_config_name = f"sapbert_full_len{MIN_FORM_LEN}" if "MIN_FORM_LEN" in globals() else "sapbert_full_len3"
if "CONFIDENCE_THRESHOLD" not in globals():
    CONFIDENCE_THRESHOLD = 0.70
if "MIN_FORM_LEN" not in globals():
    MIN_FORM_LEN = 3
if "FAISS_TOP_K" not in globals():
    FAISS_TOP_K = int(CFG_YAML["umls"]["faiss_top_k"])
if "SAPBERT_ID" not in globals():
    SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
if "_device" not in globals():
    _device = "cuda" if torch.cuda.is_available() else "cpu"


def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s


def _is_cui_string(text) -> bool:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return False
    return bool(_CUI_PAT.match(str(text).strip()))


def shannon_entropy(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    probs = np.array([c / total for c in counts.values()], dtype=float)
    return float(-np.sum(probs * np.log2(np.clip(probs, 1e-12, 1.0)))), counts


def _entropy_from_labels(labels, unassigned=UNASSIGNED):
    labels = [
        unassigned if str(lab).lower() in {"nan", "na", "none", ""} else str(lab)
        for lab in labels
    ]
    assigned = [lab for lab in labels if lab != unassigned]
    n_variants = len(labels)
    n_assigned = len(assigned)
    n_unassigned = n_variants - n_assigned
    m_accepted = max(n_variants - 1, 0)
    if n_assigned == 0:
        return {
            "n_variants": n_variants,
            "n_assigned": 0,
            "n_unassigned": n_unassigned,
            "m_accepted": m_accepted,
            "all_unassigned": True,
            "n_clusters_full": 0,
            "semantic_entropy_full": np.nan,
            "normalised_semantic_entropy_full": np.nan,
            "dominant_cluster_full": unassigned,
            "is_zero_entropy": False,
        }
    h, counts = shannon_entropy(assigned)
    h_hat = h / np.log2(n_variants) if n_variants >= 2 else np.nan
    dominant = max(counts.items(), key=lambda x: x[1])[0]
    return {
        "n_variants": n_variants,
        "n_assigned": n_assigned,
        "n_unassigned": n_unassigned,
        "m_accepted": m_accepted,
        "all_unassigned": False,
        "n_clusters_full": len(counts),
        "semantic_entropy_full": h,
        "normalised_semantic_entropy_full": h_hat,
        "dominant_cluster_full": dominant,
        "is_zero_entropy": bool(h_hat <= 1e-12) if pd.notna(h_hat) else False,
    }


_MAP_REQUIRED = {
    "instance_id", "model_name", "input_variant_id", "input_type",
    "predicted_cui", "confidence", "gold_cui_norm",
}
_mapped_ok = (
    OUT_MAPPED.is_file()
    and OUT_MAPPED.stat().st_size > 0
    and _MAP_REQUIRED.issubset(set(pd.read_csv(OUT_MAPPED, nrows=0).columns))
)
if _mapped_ok:
    # Subset test, NOT a count test. mm_assemble emits only the shards whose CSVs are still
    # on disk, so the fresh set SHRINKS as mm_prune_mapped_shards.py deletes mapped shards.
    # A `_n_map < _n_all` comparison therefore gets progressively MORE wrong as work
    # succeeds: with 40,000 instances cached and 16,000 fresh it read False and skipped the
    # remap, so shards 3 and 4 never entered the mapped file after 24 GPU-hours of work.
    _ids_map = set(pd.read_csv(OUT_MAPPED, usecols=["instance_id"])["instance_id"].astype(str))
    _ids_all = set(pd.read_csv(_MODEL_OUT, usecols=["instance_id"])["instance_id"].astype(str))
    _only_fresh = _ids_all - _ids_map
    _only_cache = _ids_map - _ids_all
    print(f"MAP CACHE: cache={len(_ids_map):,} fresh={len(_ids_all):,} "
          f"only-in-cache={len(_only_cache):,} only-in-fresh={len(_only_fresh):,}")
    if _only_fresh:
        print(f"Mapped cache stale: {len(_only_fresh):,} instance(s) present in "
              f"all_model_outputs are absent from the mapped file — remap")
        _mapped_ok = False
_reuse_mapped = _mapped_ok

# A) Build / load mapped table (predicted_cui for every row)
if _reuse_mapped:
    print(f"Loading cached mapped outputs (no SapBERT/GPU): {OUT_MAPPED}")
    df_mapped = pd.read_csv(OUT_MAPPED, low_memory=False)
    # Normalise in case of stale strings
    df_mapped["predicted_cui"] = df_mapped["predicted_cui"].map(_norm_cui)
    df_mapped["gold_cui_norm"] = df_mapped["gold_cui_norm"].map(_norm_cui)
    df_mapped["confidence"] = pd.to_numeric(df_mapped["confidence"], errors="coerce")
    print(
        f"  rows={len(df_mapped):,} models={df_mapped['model_name'].nunique()} | "
        f"UNASSIGNED={(df_mapped['predicted_cui']==UNASSIGNED).mean():.1%}"
    )
    if "is_direct_cui" in df_mapped.columns:
        print("  is_direct_cui rates (mapping-only flag; ignored for entropy):")
        print(df_mapped.groupby("model_name")["is_direct_cui"].mean().round(3).to_string())
else:
    # Full mapping path (needs P2.2–P2.3 FAISS cache + GPU for free-text rows)
    from transformers import AutoModel, AutoTokenizer

    _ksel_path = _resolve_cfg_path(CFG.get("k_selection", "outputs/rq1/k_selection.json"))
    if _ksel_path is not None and Path(_ksel_path).exists():
        with open(_ksel_path, "r", encoding="utf-8") as _f:
            TOP_K = int(json.load(_f).get("k_selected", FAISS_TOP_K))
    else:
        TOP_K = int(FAISS_TOP_K)
    print(f"TOP_K={TOP_K} | mapping cache missing — will SapBERT-map free-text rows")

    df_model_part1 = pd.read_csv(_MODEL_OUT).reset_index(drop=True)
    _prev_mapped = None
    if (
        OUT_MAPPED.is_file()
        and OUT_MAPPED.stat().st_size > 0
        and _MAP_REQUIRED.issubset(set(pd.read_csv(OUT_MAPPED, nrows=0).columns))
    ):
        _prev_mapped = pd.read_csv(OUT_MAPPED, low_memory=False)
        _have = set(_prev_mapped["instance_id"].astype(str))
        _before_n = df_model_part1["instance_id"].nunique()
        df_model_part1 = df_model_part1[
            ~df_model_part1["instance_id"].astype(str).isin(_have)
        ].reset_index(drop=True)
        print(
            f"Incremental map: keep {len(_have):,} inst already mapped; "
            f"new {df_model_part1['instance_id'].nunique():,} / {_before_n:,}"
        )
    need = ["instance_id", "model_name", "input_variant_id", "input_type",
            "output_text", "gold_cui_or_entity"]
    missing = [c for c in need if c not in df_model_part1.columns]
    assert not missing, f"rq1_all_model_outputs.csv missing: {missing}"
    if "perturbation_type" not in df_model_part1.columns:
        df_model_part1["perturbation_type"] = df_model_part1["input_type"]
    if _INST.is_file():
        _inst = pd.read_csv(_INST, usecols=["instance_id", "gold_mention"])
        df_model_part1 = df_model_part1.merge(_inst, on="instance_id", how="left")
    else:
        df_model_part1["gold_mention"] = np.nan

    def _mean_pool(last_hidden, attn_mask):
        mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def _embed_with_model(model, tokenizer, texts, batch_size=64, max_len=64, desc="embed"):
        vecs = []
        model.eval()
        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc=desc, unit="batch"):
                batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
                enc = tokenizer(
                    batch, return_tensors="pt", truncation=True, max_length=max_len, padding=True
                )
                enc = {k: v.to(_device) for k, v in enc.items()}
                use_amp = (_device == "cuda") and (next(model.parameters()).dtype == torch.float16)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    out = model(**enc)
                    pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
                    pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
                vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
        return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)

    _rej = Counter()

    def assign_with_encoder_scores(query_text, mention_text, form_scores):
        rule_path = []
        cand = []
        for form, sc in form_scores.items():
            for cui in _form_to_cuis.get(form, ()):
                cand.append((cui, form, float(sc)))
        if not cand:
            _rej["faiss_empty"] += 1
            return UNASSIGNED, 0.0, "faiss_empty"
        exact_cuis = set()
        for key in [mention_text, query_text]:
            if key and str(key).strip():
                exact_cuis |= set(_exact_index.get(str(key).strip().casefold(), ()))
        if exact_cuis:
            exact_cand = [c for c in cand if c[0] in exact_cuis]
            if exact_cand:
                cand = exact_cand
                rule_path.append("exact_match")
                _rej["exact_match_hit"] += 1
            else:
                cand = [(c, str(mention_text), 1.0) for c in exact_cuis] + cand
                rule_path.append("exact_match_inject")
                _rej["exact_match_hit"] += 1
        else:
            rule_path.append("no_exact_match")
        st_filt = [c for c in cand if _cui_st21pv.get(c[0], False)]
        if st_filt:
            cand = st_filt
            rule_path.append("st21pv")
            _rej["st21pv_kept"] += 1
        else:
            rule_path.append("st21pv_skip")
            _rej["st21pv_filtered_all"] += 1
        cand.sort(key=lambda x: x[2], reverse=True)
        rule_path.append("encoder_cosine")
        best_score = cand[0][2]
        if best_score < CONFIDENCE_THRESHOLD:
            _rej["below_confidence"] += 1
            rule_path.append(f"below_thresh_{CONFIDENCE_THRESHOLD}")
            return UNASSIGNED, best_score, "+".join(rule_path)
        top = [c for c in cand if (best_score - c[2]) <= 0.02]
        top.sort(key=lambda x: (_cui_n_forms.get(x[0], 0), x[2]), reverse=True)
        _rej["freq_tiebreak"] += 1
        _rej["assigned"] += 1
        rule_path.append("freq_tiebreak")
        return top[0][0], float(top[0][2]), "+".join(rule_path)

    _out_strs = df_model_part1["output_text"].fillna("").astype(str)
    _is_direct = _out_strs.map(_is_cui_string)
    print(f"direct-CUI rows: {_is_direct.sum():,} | free-text: {(~_is_direct).sum():,}")

    _pred_cui = [UNASSIGNED] * len(df_model_part1)
    _pred_sc = [0.0] * len(df_model_part1)
    _pred_path = [""] * len(df_model_part1)
    for i in np.flatnonzero(_is_direct.to_numpy()):
        _pred_cui[i] = _norm_cui(_out_strs.iloc[i])
        _pred_sc[i] = 1.0 if _pred_cui[i] != UNASSIGNED else 0.0
        _pred_path[i] = "direct_cui"
        _rej["direct_cui"] += 1

    _text_idx = np.flatnonzero(~_is_direct.to_numpy())
    if len(_text_idx):
        assert "_faiss_index" in globals() and "_form_embeddings" in globals(), (
            "Run P2.2–P2.3 first, or provide rq1_all_outputs_mapped.csv"
        )
        _sap_tok = AutoTokenizer.from_pretrained(SAPBERT_ID)
        _sap_mdl = AutoModel.from_pretrained(SAPBERT_ID)
        _sap_mdl.to(_device).eval()
        if _device == "cuda":
            _sap_mdl.half()
        texts = _out_strs.iloc[_text_idx].tolist()
        mentions = df_model_part1.loc[_text_idx, "gold_mention"].fillna("").astype(str).tolist()
        q_vecs = _embed_with_model(
            _sap_mdl, _sap_tok, texts, batch_size=128, max_len=64, desc="SapBERT output_text"
        )
        _D, _I = _faiss_index.search(q_vecs.astype(np.float32), TOP_K)
        for local_i, row_i in enumerate(tqdm(_text_idx, desc="five-rule assign")):
            form_scores = {}
            for sc, ix in zip(_D[local_i], _I[local_i]):
                if int(ix) < 0:
                    continue
                form = _unique_forms[int(ix)]
                if len(form) < MIN_FORM_LEN:
                    continue
                form_scores[form] = float(np.dot(q_vecs[local_i], _form_embeddings[int(ix)]))
            cui, sc, path = assign_with_encoder_scores(texts[local_i], mentions[local_i], form_scores)
            _pred_cui[row_i] = _norm_cui(cui)
            _pred_sc[row_i] = float(sc)
            _pred_path[row_i] = path
        del _sap_mdl, _sap_tok, q_vecs
        gc.collect()
        if _device == "cuda":
            torch.cuda.empty_cache()

    df_mapped = df_model_part1[
        ["instance_id", "model_name", "input_variant_id", "input_type",
         "output_text", "gold_cui_or_entity", "perturbation_type", "gold_mention"]
    ].copy()
    df_mapped["predicted_cui"] = _pred_cui
    df_mapped["confidence"] = _pred_sc
    df_mapped["assign_rule_path"] = _pred_path
    df_mapped["gold_cui_norm"] = df_mapped["gold_cui_or_entity"].map(_norm_cui)
    df_mapped["is_direct_cui"] = _is_direct.values
    if _prev_mapped is not None and len(_prev_mapped):
        df_mapped = pd.concat([_prev_mapped, df_mapped], ignore_index=True)
    df_mapped.to_csv(OUT_MAPPED, index=False)
    print(f"Wrote mapped cache → {OUT_MAPPED} | rows={len(df_mapped):,}")
    print("rej:", dict(_rej))

# Alias for any downstream cells that still expect these names
df_full_assign = df_mapped.rename(columns={
    "predicted_cui": "predicted_cui_full",
    "confidence": "confidence_full",
}).copy()
df_model_part1 = df_mapped  # keep a handle for legacy references

print("\n=== Per-model predicted_cui coverage (uniform column) ===")
for m, g in df_mapped.groupby("model_name"):
    print(
        f"  {m:28s}  rows={len(g):5d}  nunique={g['predicted_cui'].nunique():4d}  "
        f"UNASSIGNED={(g['predicted_cui']==UNASSIGNED).mean():.1%}  "
        f"null={g['predicted_cui'].isna().sum()}"
    )
assert df_mapped["predicted_cui"].notna().all(), "predicted_cui has nulls — refuse to score"

# --- acceptance filter AT THE ENTROPY STAGE -------------------------------------------------
# The mapped-outputs file carries no acceptance column and nothing downstream filtered on one,
# so a variant accepted when inference ran but rejected in the CURRENT gate verdicts still
# reached m_accepted and the cluster distribution. Measured: 3 instances / 24 rows, all shard 0,
# because rq1_validated_perturbations.csv was re-assembled after that shard's inference
# (docs/BUG_AUDIT.md, 2026-09-13). Drop them BEFORE m is computed; instances lose a variant and
# fall out via m >= 3 only if they actually fall out.
_n_before = len(df_mapped)
_is_orig = df_mapped["input_type"] == "original"
_ok = _is_orig | df_mapped["input_variant_id"].astype(str).isin(_accepted_vids)
df_mapped = df_mapped[_ok].copy()
print(f"Acceptance filter at entropy stage: dropped {_n_before - len(df_mapped):,} of "
      f"{_n_before:,} mapped rows lacking a current accepted counterpart", flush=True)
_resid = sorted(
    set(df_mapped.loc[df_mapped["input_type"] != "original", "input_variant_id"].astype(str))
    - _accepted_vids
)
assert not _resid, (
    f"ASSERT FAILED: {len(_resid):,} row(s) survive the acceptance filter without an accepted "
    f"counterpart. Examples: {_resid[:10]}"
)

# --- record WHICH gate verdicts these numbers used ------------------------------------------
import hashlib as _hashlib
_h = _hashlib.sha256()
with open(_VALIDATED_FULL, "rb") as _f:
    for _blk in iter(lambda: _f.read(1 << 20), b""):
        _h.update(_blk)
VALIDATED_SHA256 = _h.hexdigest()
_side = _VALIDATED_FULL.with_suffix(_VALIDATED_FULL.suffix + ".sha256.json")
if _side.is_file():
    _frozen = json.loads(_side.read_text()).get("sha256")
    print(f"Gate verdicts: {_VALIDATED_FULL.name} sha256={VALIDATED_SHA256[:16]}... "
          f"({'MATCHES' if _frozen == VALIDATED_SHA256 else '*** DIFFERS FROM ***'} frozen sidecar)",
          flush=True)
else:
    print(f"Gate verdicts: {_VALIDATED_FULL.name} sha256={VALIDATED_SHA256[:16]}... (no sidecar)",
          flush=True)

# B) Entropy / accuracy / mapping_confidence: UNIFORM over predicted_cui
#    (no is_direct_cui branching)
MIN_M_ACCEPTED = 3
entropy_full_rows = []
_excl_m = 0
_m_dist = Counter()
_m_dist_distinct = Counter()
_n_inc_dedup = 0

# --- variant input text, for the parallel de-duplicated m ------------------------------
# Mirrors CADEC_entropy cell10 exactly (commit 125d9d6). Byte-identical input variants
# necessarily yield the same output and therefore the same cluster, so they inflate the
# dominant cluster (pushing entropy toward 0) while also inflating m, the denominator
# log2(m+1). Back-translation is deterministic greedy and is generated in two slots per
# instance, so its siblings collide often: measured on the clean CADEC set, 27.6% of
# accepted back_translation rows duplicate a sibling, against 17.7% of accepted variants
# overall (syntactic_reordering 34.5%, controlled_paraphrase 24.4%, synonym_substitution
# 2.7%). The equivalent MedMentions rates are what this cell now measures.
# The existing columns and the existing m_accepted>=3 emission gate are UNCHANGED; the
# de-duplicated view is emitted alongside, flagged by retained_m_distinct, so the raw arm
# survives as the labelled sensitivity analysis (docs/ANALYSIS_PRECOMMIT.md section 3).
_VALIDATED_FULL = INTER_DIR / "rq1_validated_perturbations.csv"
_variant_text = {}
if _VALIDATED_FULL.is_file():
    _vt = pd.read_csv(
        _VALIDATED_FULL,
        usecols=["perturbation_id", "perturbation_text", "accepted_final"],
        low_memory=False,
    )
    # DECLARED keying for this lane: direct_id. MedMentions inference does NOT renumber, so
    # input_variant_id IS the perturbation_id (set match 99.99%). CADEC renumbers and needs
    # positional keying -- see scripts/dedup_key_regression.py, which asserts the declaration
    # against the data for both lanes.
    # Text resolves against EVERY row: acceptance decides position, not whether a text exists.
    _variant_text = dict(zip(_vt["perturbation_id"].astype(str),
                             _vt["perturbation_text"].fillna("").astype(str)))
    _accepted_vids = set(
        _vt.loc[_vt["accepted_final"].astype(str).str.lower().isin(("true", "1")),
                "perturbation_id"].astype(str)
    )
    print(f"Variant text map for dedup: {len(_variant_text):,} perturbation ids "
          f"({len(_accepted_vids):,} currently accepted)", flush=True)
    del _vt
else:
    print(f"WARNING: {_VALIDATED_FULL.name} missing — dedup columns will be NaN", flush=True)


def _variant_key(vid, itype, iid):
    """Text used to detect byte-identical variants. Originals key on their own id (there is
    one per instance-model); an unknown perturbation id keys on itself so a missing lookup can
    never collapse two genuinely different variants."""
    vid = str(vid)
    if itype == "original":
        return f"<<orig:{iid}>>"
    return _variant_text.get(vid, f"<<missing:{vid}>>")

for (inst, model), grp in df_mapped.groupby(["instance_id", "model_name"]):
    grp_ord = pd.concat(
        [grp[grp["input_type"] == "original"], grp[grp["input_type"] != "original"]],
        ignore_index=True,
    )
    labels = grp_ord["predicted_cui"].fillna(UNASSIGNED).astype(str).tolist()
    m_acc = max(len(labels) - 1, 0)
    _m_dist[m_acc] += 1
    if m_acc < MIN_M_ACCEPTED:
        _excl_m += 1
        continue

    ent = _entropy_from_labels(labels)

    # Parallel view: collapse byte-identical input variants, keeping first occurrence.
    _keys = [
        _variant_key(v, t, inst)
        for v, t in zip(grp_ord["input_variant_id"], grp_ord["input_type"])
    ]
    _seen, _keep_idx = set(), []
    for _i, _k in enumerate(_keys):
        if _k not in _seen:
            _seen.add(_k)
            _keep_idx.append(_i)
    labels_distinct = [labels[_i] for _i in _keep_idx]
    m_distinct = max(len(labels_distinct) - 1, 0)
    _m_dist_distinct[m_distinct] += 1
    ent_d = _entropy_from_labels(labels_distinct)
    retained_distinct = bool(m_distinct >= MIN_M_ACCEPTED)
    if retained_distinct:
        _n_inc_dedup += 1

    orig = grp[grp["input_type"] == "original"]
    if len(orig) == 0:
        acc = 0.0
        map_conf = np.nan
    else:
        pred = _norm_cui(orig.iloc[0]["predicted_cui"])
        gold = _norm_cui(orig.iloc[0]["gold_cui_norm"])
        acc = float(int(pred != UNASSIGNED and gold != UNASSIGNED and pred == gold))
        map_conf = float(pd.to_numeric(orig["confidence"], errors="coerce").mean())

    entropy_full_rows.append({
        "instance_id": inst,
        "model_name": model,
        "mean_accuracy_full": acc,
        "mapping_confidence": map_conf,
        "pool_config": _pool_config_name,
        **ent,
        # Parallel de-duplicated view. Same eight column names as CADEC entropy_cadec.csv.
        "m_distinct": m_distinct,
        "normalised_entropy_dedup": ent_d["normalised_semantic_entropy_full"],
        "semantic_entropy_dedup": ent_d["semantic_entropy_full"],
        "dominant_cui_dedup": ent_d["dominant_cluster_full"],
        "n_unassigned_dedup": ent_d["n_unassigned"],
        "n_duplicate_variants": len(labels) - len(labels_distinct),
        "retained_m_accepted": True,
        "retained_m_distinct": retained_distinct,
    })

_n_inst_model = sum(_m_dist.values())
_n_inc = _n_inst_model - _excl_m
print("\nMedMentions entropy inclusion (m>=3; H_norm = H / log2(m+1); ALL models via predicted_cui):")
print(f"  included={_n_inc:,}  excluded(m<3)={_excl_m:,}  total(instance×model)={_n_inst_model:,}")
for m_k in sorted(_m_dist):
    print(f"    m={m_k}: {_m_dist[m_k]:,}")
print(
    f"  Parallel dedup view: retained_m_distinct={_n_inc_dedup:,} of {_n_inc:,} emitted rows "
    f"({_n_inc - _n_inc_dedup:,} would drop below m>={MIN_M_ACCEPTED} once byte-identical "
    f"variants are collapsed). Existing columns and the existing filter are unchanged."
)
print("  m_distinct distribution (instance×model, emitted rows):")
for m_k in sorted(_m_dist_distinct):
    print(f"    m_distinct={m_k}: {_m_dist_distinct[m_k]:,}")

df_entropy_full = pd.DataFrame(entropy_full_rows)
assert len(df_entropy_full) > 0, "No entropy rows after m>=3 filter"
assert df_entropy_full["model_name"].nunique() >= 7, (
    f"Too few models in entropy: {sorted(df_entropy_full['model_name'].unique())}"
)

# Sanity: generative models must have finite H (not all-NaN)
_gen_models = [m for m in df_entropy_full["model_name"].unique()
               if m not in {"BERT-base", "BioBERT", "PubMedBERT"}]
for gm in _gen_models:
    sub = df_entropy_full.loc[
        (df_entropy_full["model_name"] == gm) & (~df_entropy_full["all_unassigned"].astype(bool)),
        "normalised_semantic_entropy_full",
    ]
    assert sub.notna().any(), f"BUG: {gm} has no finite normalised entropy — predicted_cui not scored"
    assert sub.notna().mean() > 0.9, f"BUG: {gm} mostly-NaN entropy ({sub.notna().mean():.1%} finite)"

# Optional mesh columns for encoders only (do NOT use these as primary H for generatives)
if _PART1_ENT.is_file():
    df_entropy_mesh = pd.read_csv(_PART1_ENT)
    _mesh_keep = [
        c for c in df_entropy_mesh.columns
        if c in {
            "instance_id", "model_name",
            "semantic_entropy", "normalised_semantic_entropy", "n_clusters",
            "mean_accuracy", "dominant_cluster", "cluster_distribution",
            "instability_flag", "n_variants",
        }
    ]
    df_entropy_mesh = df_entropy_mesh[_mesh_keep].rename(columns={
        "semantic_entropy": "semantic_entropy_mesh",
        "normalised_semantic_entropy": "normalised_semantic_entropy_mesh",
        "n_clusters": "n_clusters_mesh",
        "mean_accuracy": "mean_accuracy_mesh",
        "dominant_cluster": "dominant_cluster_mesh",
        "n_variants": "n_variants_mesh",
    })
    df_entropy_compare = df_entropy_full.merge(
        df_entropy_mesh, on=["instance_id", "model_name"], how="left"
    )
    print(
        f"Left-merged Part1 mesh cols (encoders only; generative mesh=NaN is expected): "
        f"{len(df_entropy_compare):,} rows"
    )
else:
    df_entropy_compare = df_entropy_full.copy()

df_entropy_compare.to_csv(_out_csv, index=False)
print(f"\nWrote {_out_csv}")
print("Models:", sorted(df_entropy_compare["model_name"].unique()))

print("\n=== Per-model mean normalised_semantic_entropy_full (PRIMARY) ===")
_scored_print = df_entropy_compare.loc[~df_entropy_compare["all_unassigned"].astype(bool)]
print(
    _scored_print.groupby("model_name")["normalised_semantic_entropy_full"]
    .agg(["count", "mean", "std", "median"])
    .round(6)
    .to_string()
)
print("\n=== Per-model mean_accuracy_full / mapping_confidence ===")
print(
    df_entropy_compare.groupby("model_name")[["mean_accuracy_full", "mapping_confidence"]]
    .mean()
    .round(4)
    .to_string()
)
print(
    "\nNOTE: normalised_semantic_entropy_mesh is NaN for generative models "
    "(Part1 mesh file has encoders only). Use normalised_semantic_entropy_full."
)
sys.stdout.flush()
df_entropy_compare.head()


In [ ]:
# PART 2: ASSERT model-specific full-pool entropy
# Fails loudly if the old bug (identical per-model means) returns.
# Means are computed on scored rows only (excl. all-UNASSIGNED).

_cmp = df_entropy_compare.copy()
if "all_unassigned" in _cmp.columns:
    _scored = _cmp[~_cmp["all_unassigned"]]
else:
    _scored = _cmp.dropna(subset=["normalised_semantic_entropy_full"])

_means = (
    _scored.groupby("model_name")["normalised_semantic_entropy_full"]
    .mean()
    .astype(float)
)
print("Per-model mean normalised_semantic_entropy_full (excl. all-UNASSIGNED):")
print(_means.round(6).to_string())
print(f"all-UNASSIGNED rows excluded: {len(_cmp) - len(_scored):,}")

if _means.nunique() == 1 and len(_means) > 1:
    raise AssertionError(
        "BUG: per-model mean full-pool entropy is IDENTICAL across models "
        f"({_means.iloc[0]:.6f}). Query/mapping is model-independent again. "
        "Check mixed CUI/text mapping — models should not share identical mean H."
    )

_vals = _means.values
_min_pair_diff = min(
    abs(float(_vals[i] - _vals[j]))
    for i in range(len(_vals)) for j in range(i + 1, len(_vals))
) if len(_vals) > 1 else float("inf")
print(f"Min pairwise |Δ mean H|: {_min_pair_diff:.6g}")
if _min_pair_diff < 1e-12:
    raise AssertionError(
        "BUG: at least two models share the same mean full-pool entropy to numerical precision."
    )

# Dominant should be non-null on scored rows; all-UNASSIGNED may be UNASSIGNED string
_dom_nan = _scored["dominant_cluster_full"].isna().mean()
if _dom_nan > 0.01:
    raise AssertionError(
        f"BUG: dominant_cluster_full is NaN for {_dom_nan:.1%} of scored rows "
        "(expected ~0; use UNASSIGNED not NA)."
    )

print("ASSERT OK: full-pool entropy is model-specific; scored dominant NaN rate "
      f"= {_dom_nan:.3%}")


# Expect all 8 models in the MedMentions entropy file when all-model outputs were used
# 3 encoders + 4 generatives in rq1_all_model_outputs.csv (FLAN not run on MedMentions)
_EXPECTED_MODELS = {
    "BERT-base", "BioBERT", "PubMedBERT",
    "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
    "FLAN-T5-base",
}
_have = set(_means.index)
_missing_models = sorted(_EXPECTED_MODELS - _have)
if _missing_models:
    print(f"WARNING: entropy file missing models: {_missing_models}")
else:
    print(f"ASSERT OK: all {len(_EXPECTED_MODELS)} expected models present in entropy file")


## P2.4b) Top-k sensitivity (\(k \in \{10,50,100,250\}\)): legacy encoder path

> **Note:** P2.4 now uses mixed CUI / free-text mapping on `rq1_all_model_outputs.csv`
> (direct CUI for encoder rows; SapBERT on `output_text` for generative rows).
> This top-k cell still expects the **legacy** encoder `input_text` re-rank globals
> (`_model_hf`, `QUERY_TEXT_COLUMN`, …). Re-run only if those are reconstructed;
> otherwise skip to P2.5.

Reports at each \(k\):
- per-model mean normalised \(H\), % exactly zero, between-model separation


In [ ]:
# PART 2: Top-k sensitivity (k = 10, 50, 100, 250)
print('SKIP: top-k sweep disabled (k=1000 locked as fixed external input)')


## P2.5) Comparison: mesh_subset vs full_umls entropy

Side-by-side histograms, summary statistics, **per-model** mesh↔full
zero/nonzero transition matrices (pooled matrix shown only as reference),
and an explicit check of whether Part 1 bimodality persists.

All-UNASSIGNED instances are reported separately and excluded from % zero / mean H.


In [ ]:
# PART 2: mesh vs full_umls comparison
import matplotlib.pyplot as plt

assert "df_entropy_compare" in globals(), "Run the CUI assignment / entropy cell first."

# Align UNASSIGNED / all_unassigned columns if present from patched P2.4
if "all_unassigned" not in df_entropy_compare.columns:
    # Infer: NaN full entropy ⇒ all-unassigned under patched scheme
    df_entropy_compare["all_unassigned"] = (
        df_entropy_compare["normalised_semantic_entropy_full"].isna()
    )

_scored = df_entropy_compare[~df_entropy_compare["all_unassigned"]].copy()
print(
    f"Comparison rows: {len(df_entropy_compare):,} | "
    f"scored (excl. all-UNASSIGNED): {len(_scored):,} | "
    f"all-UNASSIGNED: {df_entropy_compare['all_unassigned'].sum():,}"
)

_summary_rows = []
for label, col, df_ in [
    ("mesh_subset", "normalised_semantic_entropy_mesh", df_entropy_compare),
    ("full_umls_scored", "normalised_semantic_entropy_full", _scored),
]:
    s = df_[col].astype(float).dropna()
    _summary_rows.append({
        "pool": label,
        "n": int(s.shape[0]),
        "mean": float(s.mean()) if len(s) else np.nan,
        "median": float(s.median()) if len(s) else np.nan,
        "sd": float(s.std(ddof=1)) if len(s) > 1 else np.nan,
        "pct_exactly_zero": float((s <= 1e-12).mean() * 100) if len(s) else np.nan,
        "pct_gt_0_20": float((s > 0.20).mean() * 100) if len(s) else np.nan,
    })
_summary_rows.append({
    "pool": "full_umls_all_unassigned",
    "n": int(df_entropy_compare["all_unassigned"].sum()),
    "mean": np.nan, "median": np.nan, "sd": np.nan,
    "pct_exactly_zero": np.nan, "pct_gt_0_20": np.nan,
})

df_entropy_summary = pd.DataFrame(_summary_rows)
print("=== Normalised semantic entropy summary ===")
display(df_entropy_summary)

print("\n=== Per-model means (normalised H; full excl. all-UNASSIGNED) ===")
display(
    _scored.groupby("model_name")[
        ["normalised_semantic_entropy_mesh", "normalised_semantic_entropy_full"]
    ].mean().round(4)
)

# Per-model mesh↔full zero/nonzero transition matrices (NOT pooled)
def _zf(s):
    out = pd.Series(np.nan, index=s.index, dtype=float)
    v = s.notna()
    out.loc[v] = (s.loc[v].astype(float) <= 1e-12).astype(float)
    return out

_tm = df_entropy_compare.copy()
_tm = _tm[~_tm["all_unassigned"]]
_tm["mesh_zero"] = _zf(_tm["normalised_semantic_entropy_mesh"])
_tm["full_zero"] = _zf(_tm["normalised_semantic_entropy_full"])
_tm = _tm.dropna(subset=["mesh_zero", "full_zero"])

print("\n=== Per-model transition matrices: mesh zero/nonzero → full zero/nonzero ===")
_trans_rows = []
for model, g in _tm.groupby("model_name"):
    mat = pd.crosstab(
        g["mesh_zero"].map({1.0: "mesh_zero", 0.0: "mesh_nonzero"}),
        g["full_zero"].map({1.0: "full_zero", 0.0: "full_nonzero"}),
        dropna=False,
    )
    # ensure all 4 cells present
    for r in ["mesh_zero", "mesh_nonzero"]:
        if r not in mat.index:
            mat.loc[r] = 0
    for c in ["full_zero", "full_nonzero"]:
        if c not in mat.columns:
            mat[c] = 0
    mat = mat.reindex(index=["mesh_zero", "mesh_nonzero"],
                      columns=["full_zero", "full_nonzero"]).fillna(0).astype(int)
    print(f"\n{model} (n={len(g)}):")
    display(mat)
    _trans_rows.append({
        "model_name": model,
        "n": int(len(g)),
        "zero_to_zero": int(mat.loc["mesh_zero", "full_zero"]),
        "zero_to_nonzero": int(mat.loc["mesh_zero", "full_nonzero"]),
        "nonzero_to_zero": int(mat.loc["mesh_nonzero", "full_zero"]),
        "nonzero_to_nonzero": int(mat.loc["mesh_nonzero", "full_nonzero"]),
    })

df_mesh_full_transitions = pd.DataFrame(_trans_rows)
print("\n(Pooled reference — do not use for claims; per-model above is primary)")
_pooled = pd.crosstab(
    _tm["mesh_zero"].map({1.0: "mesh_zero", 0.0: "mesh_nonzero"}),
    _tm["full_zero"].map({1.0: "full_zero", 0.0: "full_nonzero"}),
)
display(_pooled)

_tab_dir = Path("..") / "outputs" / "rq1" / "tables"
if not _tab_dir.exists():
    _tab_dir = Path("outputs") / "rq1" / "tables"
_tab_dir.mkdir(parents=True, exist_ok=True)
df_mesh_full_transitions.to_csv(
    _tab_dir / "rq1_mesh_full_zero_nonzero_transitions_per_model.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, col, title, color, df_ in [
    (axes[0], "normalised_semantic_entropy_mesh",
     "Part 1: mesh_subset (~1.2k CUIs)", "#4C72B0", df_entropy_compare),
    (axes[1], "normalised_semantic_entropy_full",
     f"Part 2: full_umls scored ({_pool_config_name})", "#DD8452", _scored),
]:
    vals = df_[col].astype(float).dropna().values
    ax.hist(vals, bins=20, color=color, edgecolor="white", range=(0, 1))
    if len(vals):
        ax.axvline(np.mean(vals), color="black", ls="--", lw=1, label=f"mean={np.mean(vals):.3f}")
        ax.axvline(np.median(vals), color="gray", ls=":", lw=1, label=f"median={np.median(vals):.3f}")
    ax.set_title(title)
    ax.set_xlabel("Normalised semantic entropy Ĥ")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
plt.tight_layout()
_fig_path = _out_dir / "fig_entropy_mesh_vs_full_umls.png"
plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure -> {_fig_path.resolve()}")

_mesh = df_entropy_compare["normalised_semantic_entropy_mesh"].astype(float)
_full = _scored["normalised_semantic_entropy_full"].astype(float)

def _bimodality_note(name, s):
    pct0 = (s <= 1e-12).mean() * 100
    pct_mid = ((s > 0.05) & (s <= 0.20)).mean() * 100
    pct_hi = (s > 0.20).mean() * 100
    persists = (pct0 >= 40) and (pct_hi >= 15) and (pct_mid < pct_hi)
    print(f"{name}: %≈0={pct0:.1f} | % (0.05,0.20]={pct_mid:.1f} | %>0.20={pct_hi:.1f}")
    print(f"  Bimodality heuristic {'PERSISTS' if persists else 'DOES NOT PERSIST'} "
          f"(zero-spike≥40% AND unstable-tail≥15% AND middle < tail).")
    return persists

print("\n=== Bimodality check ===")
_bimod_mesh = _bimodality_note("mesh_subset", _mesh)
_bimod_full = _bimodality_note("full_umls (scored)", _full)
print(
    "\nConclusion: under the ~2,600× deeper candidate pool, Part 1-style bimodality "
    + ("still appears." if _bimod_full else "does not clearly persist.")
)


## P2.6) Placeholder: `PREFERRED_ONLY` sensitivity rerun

Set `PREFERRED_ONLY = True` in the pool-preparation cell and re-run **P2.1 → P2.5**.
That rebuilds a separate embedding/FAISS cache (`sapbert_pref_tty_len{MIN_FORM_LEN}`)
and should append / display preferred-only summary numbers beside the primary
full-vocabulary results below.

Primary configuration remains **full vocabulary + length filter** (synonym recall
matters; preferred-only is sensitivity only).


In [ ]:
# PART 2: PREFERRED_ONLY sensitivity placeholder
# After rerunning P2.1–P2.5 with PREFERRED_ONLY=True, store the summary row here.
# This cell does not recompute embeddings; it only records / displays a slot.

if "df_entropy_summary" in globals():
    print("Current (active config) summary:")
    display(df_entropy_summary)
else:
    print("No df_entropy_summary yet — run P2.5 first.")

# Manual append slot (filled after preferred-only rerun):
_preferred_only_summary_row = {
    "pool": "preferred_only_PT_PN",
    "n": None,
    "mean": None,
    "median": None,
    "sd": None,
    "pct_exactly_zero": None,
    "pct_gt_0_20": None,
    "note": "Re-run P2.1–P2.5 with PREFERRED_ONLY=True, then paste values here.",
}
print("\nPreferred-only placeholder row (fill after sensitivity rerun):")
display(pd.DataFrame([_preferred_only_summary_row]))

# Optional: if a preferred-only comparison CSV was saved under a distinct name, load it.
_pref_csv = _out_dir / "entropy_full_umls_preferred_only.csv"
if _pref_csv.exists():
    _df_pref = pd.read_csv(_pref_csv)
    _s = _df_pref["normalised_semantic_entropy_full"].astype(float)
    _row = {
        "pool": "preferred_only_PT_PN",
        "n": int(_s.shape[0]),
        "mean": float(_s.mean()),
        "median": float(_s.median()),
        "sd": float(_s.std(ddof=1)),
        "pct_exactly_zero": float((_s <= 1e-12).mean() * 100),
        "pct_gt_0_20": float((_s > 0.20).mean() * 100),
    }
    display(pd.concat([df_entropy_summary, pd.DataFrame([_row])], ignore_index=True))
else:
    print(f"(No {_pref_csv.name} yet — after preferred-only rerun, save compare CSV there.)")


## P2.7) Extended top-k sweep (batch / nbconvert-safe) + k-selection

**Kernel prerequisites (from P2.1–P2.4, not Setup alone):**
`_faiss_index`, `_embed_with_model`, `_unique_forms`, `df_model_part1`,
`CAND_GEN_COLUMN`, `assign_with_encoder_scores`, `_entropy_from_labels`,
`_model_hf`, `QUERY_TEXT_COLUMN`, `EXACT_MATCH_COLUMN`, `UNASSIGNED`, `_norm_cui`,
`_device`, `SAPBERT_ID`.

**Batch design:** one FAISS pass at \(k_{\max}=1000\) (checkpointed); depths are
prefix slices. Re-rank order is fail-fast: **250 first** → integrity assert → then
250, 500, 1000, 100, 50, 10 (existing per-(model,k) CSVs are skipped on resume).


### P2.7a) Checkpointed FAISS retrieval at \(k_{\max}=1000\)

Writes/loads `outputs/rq1/intermediate/faiss_shortlist_{I,D}_k1000.npy` and
`faiss_shortlist_query_texts.json`. Skips search if all three files already exist.


In [ ]:
# PART 2: Extended top-k sweep: Cell 1: FAISS k_max=1000 (checkpointed)
print('SKIP: top-k sweep disabled (k=1000 locked as fixed external input)')


### P2.7b) Per-model re-rank (fail-fast k=250, then resumable sweep)

Order: **250 → assert → 250, 500, 1000, 100, 50, 10**. Skips existing non-empty
`rq1_topk_sweep_{Model}_k{k}.csv`. Embeds unique shortlist forms once per encoder.


In [ ]:
# PART 2: Per-model re-rank (fail-fast + resumable)
print('SKIP: top-k sweep disabled (k=1000 locked as fixed external input)')


### P2.7c) Integrity assertions (post-sweep)

(b) Distinct per-model means at every \(k\).  
(c) Prefix nesting on `_I_kmax`.  
(d) Per-model **`all_unassigned`** counts (instances where all variants are unassigned).


In [ ]:
# PART 2: Integrity assertions (extended top-k sweep)
print('SKIP: top-k sweep disabled (k=1000 locked as fixed external input)')


### P2.7d) k-selection from disk → `outputs/rq1/k_selection.json`

Rebuilds `rq1_topk_sweep_full.csv` by reading per-(model,k) CSVs from disk (resume-safe).
SE reported at **every** \(k\); k=250 is the fixed SE reference (±20% warning at selected k).


In [ ]:
# PART 2: k-selection static record (sweep locked)
# k=1000 is LOCKED. Do not re-run the top-k sweep. Rewrite k_selection.json
# only as a static provenance record.
from datetime import datetime, timezone
import json as _json

_ksel_path = PROJECT_ROOT / "outputs" / "rq1" / "k_selection.json"
_ksel_path.parent.mkdir(parents=True, exist_ok=True)
_k_selection = {
    "k_selected": int(CFG_YAML["umls"]["faiss_top_k"]),
    "rule_satisfied": False,
    "locked": True,
    "notes": (
        "k=1000 locked for the full-corpus rerun. Historical sweep CSVs are not "
        "regenerated. This file is a static record, not a new selection."
    ),
    "written_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
}
if _ksel_path.is_file():
    _prev = _json.loads(_ksel_path.read_text(encoding="utf-8"))
    print(f"existing k_selection.json k_selected={_prev.get('k_selected')} locked={_prev.get('locked')}")
    if int(_prev.get("k_selected", -1)) == int(_k_selection["k_selected"]):
        print(f"keeping {_ksel_path} (already k={_k_selection['k_selected']})")
    else:
        _ksel_path.write_text(_json.dumps(_k_selection, indent=2) + "\n", encoding="utf-8")
        print(f"rewrote static record {_ksel_path}")
else:
    _ksel_path.write_text(_json.dumps(_k_selection, indent=2) + "\n", encoding="utf-8")
    print(f"wrote static record {_ksel_path}")

# config.json is a fixed input; k_selection path is already set there.
print("config.json left unchanged")
